# Evolution of phenotypic plasticity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/zoo/phenotypic_plasticity.ipynb)

*Syga S, Jain HP, Krellner M, Hatzikirou H, Deutsch A (2024). Evolution of
phenotypic plasticity leads to tumor heterogeneity with implications for
therapy. PLoS Comput Biol 20(8): e1012003.
[doi:10.1371/journal.pcbi.1012003](https://doi.org/10.1371/journal.pcbi.1012003)*

**Question.** Tumour cells switch between migrating and dividing, and how
they switch may itself be inherited and mutate. When every cell carries its
own switch, which switching strategies does evolution favour in a growing
tumour, and where in the tumour do they end up?

The notebook reproduces the three evolutionary regimes of the paper: in
two dimensions (S1–S3 Figs), and in one dimension at the paper's size
(Fig 3 A–F).

In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import lgca
from lgca import zoo
from lgca.model import run_model
from lgca.zoo import phenotypic_plasticity as plasticity

zoo.parameter_table("phenotypic_plasticity")

## The model

Identity-based cells without volume exclusion on a hexagonal lattice; a
node holds any number of cells, and division slows down as the node fills
up to its carrying capacity $K$. Cells in the rest channel have the
proliferative phenotype, cells in velocity channels the migratory one.
Every cell carries its own switch parameter $\kappa$, its genotype. In
every time step:

1. **Death:** every cell dies with probability $\delta$.
2. **Switch:** every cell takes the proliferative phenotype with
   probability
   $$r_\kappa(\rho_N) = \tfrac12\left(1 + \tanh\big(\kappa(\rho_N - \theta)\big)\right)$$
   and the migratory one otherwise, where $\rho_N$ is the number of cells
   on the node and its neighbours over $(b + 1)K$.
3. **Division:** proliferative cells divide with probability
   $\alpha(1 - n/K)$; the daughter rests and inherits $\kappa$ with a
   normal change, $\kappa_d \sim \mathcal N(\kappa_m, \Delta\kappa^2)$.
4. **Migration:** migratory cells pick a random direction and move.

$\kappa > 0$ is the *attractive* strategy (divide where it is crowded,
migrate where it is sparse), $\kappa < 0$ the *repulsive* one (migrate away
from crowding), $\kappa \approx 0$ the *independent* one. The tumour starts
from $K$ cells at the centre whose $\kappa$ is uniform on $[-4, 4]$. The
three regimes of the paper differ in the threshold $\theta$ and the death
rate $\delta$:

In [ ]:
print(plasticity.REGIMES)
for operator in plasticity.regime(2).dynamics.operators:
    print(operator)

## Three regimes in two dimensions (S1–S3 Figs)

One run per regime on a 120 × 120 hexagonal lattice for 200 steps (the
paper: 250 × 250 and 300 steps, `build_spec(full=True)`). The maps show all
cells, the migratory and the proliferative ones per node, and the mean
$\kappa$ of the cells at each node:

In [ ]:
runs = {number: run_model(plasticity.regime(number), showprogress=False) for number in (1, 2, 3)}

fig, axes = plt.subplots(3, 4, figsize=(15, 11), constrained_layout=True)
titles = {"cells": "all cells", "migrating": "migratory", "resting": "proliferative", "kappa": "mean κ"}
for row, (number, run) in enumerate(runs.items()):
    lattice = run.lgca
    maps = plasticity.node_maps(lattice)
    for column, (name, values) in enumerate(maps.items()):
        axis = axes[row, column]
        options = ({"cmap": "RdBu_r", "vmin": -4, "vmax": 4, "cbarlabel": "κ"} if name == "kappa"
                   else {"cmap": "viridis", "vmin": 0, "vmax": 60, "cbarlabel": "cells"})
        lattice.plot_scalarfield(values, ax=axis, tight_layout=False, mask=maps["cells"] == 0, **options)
        x, y = lattice.xcoords[maps["cells"] > 0], lattice.ycoords[maps["cells"] > 0]
        axis.set(xlim=(x.min() - 2, x.max() + 2), ylim=(y.min() - 2, y.max() + 2))  # the colony only
        axis.set_title(f"regime {number}: {titles[name]}")
plt.show()
plt.close(fig)

The three regimes of the paper:

- **Regime 1**, no death ($\theta = 0.5$, $\delta = 0$): the independent
  strategy ($\kappa \approx 0$) takes over the whole tumour, and many cells
  migrate. Without death there is no cost to migrating, and cells that
  switch regardless of their neighbours reach free space first.
- **Regime 2**, death and a low threshold ($\theta = 0.2$, $\delta = 0.2$):
  a dense core of attractive cells ($\kappa > 0$, red) surrounded by a
  sparse rim of repulsive cells ($\kappa < 0$, blue). In the crowded core
  cells wait, dividing into gaps; at the rim, cells that migrate away from
  crowding reach empty space.
- **Regime 3**, death and a high threshold ($\theta = 0.9$, $\delta = 0.2$):
  repulsive cells throughout, with weaker switches ($\kappa$ closer to zero)
  at the rim.

Against the distance from the centre (the analogue of Fig 3 G–L; the
distance is in units of the front, the radius within which 95 % of the
occupied nodes lie):

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True, constrained_layout=True)
for column, (number, run) in enumerate(runs.items()):
    profile = plasticity.radial_profiles(run.lgca)
    r = profile["r"]
    axes[0, column].plot(r, profile["resting"], color="tab:blue", label="proliferative")
    axes[0, column].plot(r, profile["migrating"], color="tab:orange", label="migratory")
    axes[0, column].set(title=f"regime {number}", ylabel="cells per node")
    mean, std = profile["kappa_mean"], profile["kappa_std"]
    axes[1, column].plot(r, mean, color="k")
    axes[1, column].fill_between(r, mean - std, mean + std, color="grey", alpha=0.3)
    axes[1, column].axhline(0, color="grey", lw=0.8)
    axes[1, column].set(xlabel="distance from the centre / front", ylabel="κ (mean ± sd)")
axes[0, 0].legend()
plt.show()
plt.close(fig)

for number, run in runs.items():
    summary = plasticity.core_and_rim(run.lgca)
    print(f"regime {number}: mean κ in the core {summary['core']:+.2f}, at the rim {summary['rim']:+.2f}, "
          f"migratory {summary['migrating']:.0%}")

The core is the inner half of the cells, the rim the outermost tenth.

## One dimension, at the paper's size (Fig 3 A–F)

The paper's main runs are one-dimensional: 1001 nodes, $K = 100$, 1000
steps, cheap enough to run here as in the paper. The kymographs show time
running down, the cells per node (top) and the mean $\kappa$ per node
(bottom):

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), constrained_layout=True)
for column, number in enumerate((1, 2, 3)):
    record = plasticity.kymographs(plasticity.regime(number, geometry="lin"), every=10)
    extent = (-0.5, record["cells"].shape[1] - 0.5, record["steps"][-1], 0)
    image = axes[0, column].imshow(record["cells"], aspect="auto", extent=extent, cmap="viridis",
                                   interpolation="nearest")
    fig.colorbar(image, ax=axes[0, column], label="cells")
    image = axes[1, column].imshow(record["kappa"], aspect="auto", extent=extent, cmap="RdBu_r", vmin=-6, vmax=6,
                                   interpolation="nearest")
    fig.colorbar(image, ax=axes[1, column], label="mean κ")
    axes[0, column].set(title=f"regime {number}", ylabel="time step $k$")
    axes[1, column].set(xlabel="$x$", ylabel="time step $k$")
    summary = plasticity.core_and_rim(record["lgca"])
    print(f"regime {number} (1D): mean κ in the core {summary['core']:+.2f}, at the rim {summary['rim']:+.2f}")
plt.show()
plt.close(fig)

The same three regimes: independent cells spreading fast (1), an
attractive bulk behind a thin repulsive front (2), and repulsive cells
throughout (3). In 1D the front of regime 2 is only a few dozen nodes wide
and strongly repulsive ($\kappa$ around $-3$ to $-7$), while the bulk is
strongly attractive.

## Differences from the paper

- **Size in 2D.** 120 × 120 nodes and 200 steps instead of 250 × 250 and
  300; `build_spec(full=True)` gives the paper's. The 1D runs have the
  paper's size.
- **Single runs.** The paper's Fig 3 G–L averages over realizations; the
  profiles above are single runs. Other seeds give the same picture (the
  test suite checks two).
- **Order of the rules.** Death, switch, division, migration, as in the
  paper. The research model `go_or_grow_kappa` of earlier biolgca versions
  switches before death.
- **Boundaries.** Reflecting; the tumours do not reach them.

## Explore it

A smaller tumour, with the mean $\kappa$ per node and its mean over all
cells beside it. Move $\theta$ and $\delta$ (`death_rate`) while it grows and watch the
strategies change (**Reset** starts again with new initial cells):

In [ ]:
explorer = lgca.explore(
    plasticity.regime(2, size=80),
    {"theta": (0.0, 1.0), "death_rate": (0.0, 0.3)},  # θ and δ
    view="mean kappa",
    measure=["population", "kappa"],
    steps_per_frame=2,
)
explorer

## Things to try

1. **Therapy.** The paper removes 99.9 % of the cells at the end of a run
   and measures the time until the tumour regrows. Write a rule that does
   this once (the how-to *Custom interactions* shows how), add it to the
   pipeline, and compare the regimes: which recovers fastest, and why?
2. **Phase diagram (Fig 4).** Sweep $\theta$ and $\delta$ in 1D with
   `lgca.study.sweep` and measure the mean $\kappa$ of all cells, e.g.
   `measure={"kappa": lambda result: plasticity.core_and_rim(result.lgca)["all"]}`.
   Where do the three regimes lie?
3. **No mutation.** Set $\Delta\kappa = 0$: the initial variety of
   $\kappa$ is all selection has to work with. Does the outcome change?
4. **The neighbourhood.** Use the density of the node alone
   (`"density": "node"` in `go_or_rest`) instead of the neighbourhood.